# فصل ۱۳: پردازش داده‌ها در LangChain (Data Processing)
## بخش ۴: هنر حفظ بافتار، همپوشانی (Chunk Overlap) و متادیتای زمینه‌ای

### 🕵️ تراژدی «ضمیرهای سرگردان» (The Dangling Pronoun Tragedy)
حتی اگر از پیشرفته‌ترین اسپلیترها استفاده کنید، عمل برش زدن ذاتاً یک ریسک دارد: **قطع شدن پیوند معنایی میان جملات مجاور**.

داستان کوتاه زیر را در نظر بگیرید:
> *«قاتل وارد اتاق شد. علی در را بست. او اسلحه را زیر مبل پنهان کرد.»*

اگر اسپلیتر متن را دقیقاً در مرز جملات بدون همپوشانی خرد کند:
- **چانک ۱:** «قاتل وارد اتاق شد. علی در را بست.»
- **چانک ۲:** «او اسلحه را زیر مبل پنهان کرد.»

حال کاربر می‌پرسد: *«چه کسی اسلحه را پنهان کرد؟»*
سیستم RAG چانک ۲ را بازیابی می‌کند زیرا کلمات «اسلحه» و «پنهان» در آن قرار دارند. مدل زبانی چانک را می‌خواند: *«او اسلحه را...»*؛ اما چانک ۱ در حافظه مدل نیست! مدل نمی‌داند مرجع ضمیر «او» علی است یا قاتل! نتیجه؟ **شکست در پاسخ یا توهم خطرناک (Hallucination)**.

در این بخش با دو راهکار مهندسی برای حفظ بافتار آشنا می‌شویم:
1. **همپوشانی چانک‌ها (Chunk Overlap)**
2. **متادیتای بافتی و سلسله‌مراتبی (Contextual Metadata)**

### ۱. تکنیک همپوشانی (Chunk Overlap): پل ارتباطی میان چانک‌ها

همپوشانی یعنی ایجاد یک **ناحیه مشترک** میان دو قطعه متوالی.
اگر `chunk_size=100` و `chunk_overlap=20` باشد:
- چانک اول از کاراکتر ۰ تا ۱۰۰ را پوشش می‌دهد.
- چانک دوم به جای شروع از ۱۰۱، از کاراکتر **۸۰** شروع شده و تا ۱۸۰ ادامه می‌یابد.

> 🏃‍♂️ **تمثیل دویدن امدادی:** دونده دوم پیش از آنکه دونده اول متوقف شود، چند متر در کنار او می‌دود تا چوب امدادی را تحویل بگیرد؛ آن مسافت مشترک، همان **Overlap** است که کانتکست را از قطعه قبل به بعد منتقل می‌کند.

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = "قاتل وارد اتاق شد. علی در را بست. او اسلحه را زیر مبل پنهان کرد."

# اسپلیتر بدون همپوشانی (خطر قطع مرجع ضمیر)
splitter_bad = RecursiveCharacterTextSplitter(
    chunk_size=25,
    chunk_overlap=0
)

# اسپلیتر همراه با همپوشانی (انتقال کانتکست)
splitter_good = RecursiveCharacterTextSplitter(
    chunk_size=25,
    chunk_overlap=10
)

print("--- ۱. بدون همپوشانی (chunk_overlap=0) ---")
for doc in splitter_bad.create_documents([text]):
    print(f"[{doc.page_content}]")

print("\n--- ۲. با همپوشانی (chunk_overlap=10) ---")
for doc in splitter_good.create_documents([text]):
    print(f"[{doc.page_content}]")

--- ۱. بدون همپوشانی (chunk_overlap=0) ---
[قاتل وارد اتاق شد. علی در]
[را بست. او اسلحه را زیر]
[مبل پنهان کرد.]

--- ۲. با همپوشانی (chunk_overlap=10) ---
[قاتل وارد اتاق شد. علی در]
[علی در را بست. او اسلحه]
[او اسلحه را زیر مبل]
[زیر مبل پنهان کرد.]


#### چه مقدار Overlap مناسب است؟
- **اگر خیلی کم باشد (۰٪ تا ۵٪):** خطر گم شدن ضمایر و نصف شدن مفاهیم همچنان وجود دارد.
- **اگر خیلی زیاد باشد (بالای ۴۰٪ یا ۵۰٪):** دیتابیس برداری با داده‌های تکراری باد می‌کند و ارسال چانک‌های تکراری به مدل باعث سردرگمی یا هزینه اضافه می‌شود.
- **استاندارد طلایی صنعت:** معمولاً **۱۰٪ تا ۲۰٪** از اندازه چانک (مثلاً برای `chunk_size=1000`، همپوشانی `chunk_overlap=150` تا `200` ایده‌آل است).

### ۲. ارث‌بری متادیتا: پیشگیری از یتیم شدن چانک‌ها (Metadata Inheritance)

وقتی یک فایل PDF صد صفحه‌ای را به ۵۰۰ چانک خرد می‌کنیم، اگر متادیتا منتقل نشود، چانک شماره ۲۵۰ یک **«متن بی‌هویت»** خواهد بود که مشخص نیست از چه سندی و از کدام صفحه آمده است.

در LangChain، اگر به جای متن خام از متد **`split_documents`** استفاده کنید، دیکشنری متادیتای سند والد به صورت خودکار به تمام چانک‌های فرزند ارث می‌رسد:

In [2]:
from langchain_core.documents import Document

# سندی که فرضا از PyPDFLoader بارگذاری شده است
parent_doc = Document(
    page_content="هوش مصنوعی شاخه‌ای گسترده از علوم رایانه است که کاربردهای فراوانی در پزشکی، آموزش و تجارت دارد. " * 10,
    metadata={
        "source": "AI_Handbook.pdf",
        "page": 14,
        "author": "Dr. Ahmadi",
        "category": "Computer Science"
    }
)

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=30)

# خرد کردن لیست اسناد (نه فقط رشته متنی)
child_chunks = splitter.split_documents([parent_doc])

print(f"تعداد چانک‌های تولیدشده از سند والد: {len(child_chunks)}\n")
for i, child in enumerate(child_chunks[:3], 1):
    print(f"چانک فرزند {i}:")
    print(f"  متن: {child.page_content[:60]}...")
    print(f"  متادیتا (کپی‌شده از والد): {child.metadata}\n")

تعداد چانک‌های تولیدشده از سند والد: 10

چانک فرزند 1:
  متن: هوش مصنوعی شاخه‌ای گسترده از علوم رایانه است که کاربردهای فر...
  متادیتا (کپی‌شده از والد): {'source': 'AI_Handbook.pdf', 'page': 14, 'author': 'Dr. Ahmadi', 'category': 'Computer Science'}

چانک فرزند 2:
  متن: دارد. هوش مصنوعی شاخه‌ای گسترده از علوم رایانه است که کاربرد...
  متادیتا (کپی‌شده از والد): {'source': 'AI_Handbook.pdf', 'page': 14, 'author': 'Dr. Ahmadi', 'category': 'Computer Science'}

چانک فرزند 3:
  متن: و تجارت دارد. هوش مصنوعی شاخه‌ای گسترده از علوم رایانه است ک...
  متادیتا (کپی‌شده از والد): {'source': 'AI_Handbook.pdf', 'page': 14, 'author': 'Dr. Ahmadi', 'category': 'Computer Science'}



### ۳. تکنیک پیشرفته: متادیتای بافتی با `MarkdownHeaderTextSplitter`

گاهی اوقات حتی Overlap هم نمی‌تواند تمام زمینه را حفظ کند. سناریوی پزشکی زیر را ببینید:
- بخش ۱: *«درمان سردرد: باید استامینوفن مصرف شود.»*
- بخش ۲: *«درمان دل‌درد: نباید استامینوفن مصرف شود.»*

اگر چانکی فقط حاوی عبارت *«نباید استامینوفن مصرف شود»* باشد، بسیار خطرناک است زیرا مشخص نیست مربوط به کدام بیماری است!

اسپلیتر **`MarkdownHeaderTextSplitter`** متون مارک‌داون را بر اساس تیترهای سلسله‌مراتبی (`#`, `##`, `###`) خرد می‌کند و عنوان هر تیتر را مستقیماً به **دیکشنری Metadata** همان چانک تزریق می‌نماید:

In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

medical_markdown = """
# راهنمای دارویی

## سردرد
برای تسکین درد خفیف سر، بیمار می‌تواند قرص استامینوفن استفاده کند.

## دل‌درد
در دردهای ناحیه شکمی نباید قرص استامینوفن بدون مشورت پزشک مصرف شود.
"""

# تعریف استراتژی تیترها:
# تیتر سطح ۱ (#) به عنوان category و تیتر سطح ۲ (##) به عنوان sickness ذخیره شود
headers_to_split_on = [
    ("#", "category"),
    ("##", "sickness"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_chunks = md_splitter.split_text(medical_markdown)

for i, chunk in enumerate(md_chunks, 1):
    print(f"--- چانک {i} ---")
    print(f"متن:     '{chunk.page_content.strip()}'")
    print(f"متادیتا: {chunk.metadata}\n")

--- چانک 1 ---
متن:     'برای تسکین درد خفیف سر، بیمار میتواند قرص استامینوفن استفاده کند.'
متادیتا: {'category': 'راهنمای دارویی', 'sickness': 'سردرد'}

--- چانک 2 ---
متن:     'در دردهای ناحیه شکمی نباید قرص استامینوفن بدون مشورت پزشک مصرف شود.'
متادیتا: {'category': 'راهنمای دارویی', 'sickness': 'دلدرد'}



### ۴. ترکیب حرفه‌ای دو مرحله‌ای (Two-Stage Chunking Pipeline)

در پروژه‌های واقعی اسناد مارک‌داون ابتدا ساختار دارند اما ممکن است محتوای زیر یک تیتر خیلی طولانی باشد.
معماری ایده‌آل به این صورت است:
1. **مرحله اول:** استفاده از `MarkdownHeaderTextSplitter` برای دسته‌بندی و تزریق تیترها در متادیتا.
2. **مرحله دوم:** استفاده از `RecursiveCharacterTextSplitter` روی خروجی مرحله اول برای خرد کردن بخش‌های طولانی همراه با همپوشانی (Overlap).

In [4]:
# خط لوله دو مرحله‌ای (Two-Stage Chunking)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=60,
    chunk_overlap=15
)

# اعمال مرحله دوم بر روی چانک‌های مرحله اول
final_chunks = text_splitter.split_documents(md_chunks)

print(f"تعداد نهایی چانک‌ها در خط لوله دو مرحله‌ای: {len(final_chunks)}\n")
for i, c in enumerate(final_chunks, 1):
    print(f"چانک {i}:")
    print(f"  متن:     {c.page_content}")
    print(f"  متادیتا: {c.metadata}\n")

تعداد نهایی چانک‌ها در خط لوله دو مرحله‌ای: 4

چانک 1:
  متن:     برای تسکین درد خفیف سر، بیمار میتواند قرص استامینوفن استفاده
  متادیتا: {'category': 'راهنمای دارویی', 'sickness': 'سردرد'}

چانک 2:
  متن:     استفاده کند.
  متادیتا: {'category': 'راهنمای دارویی', 'sickness': 'سردرد'}

چانک 3:
  متن:     در دردهای ناحیه شکمی نباید قرص استامینوفن بدون مشورت پزشک
  متادیتا: {'category': 'راهنمای دارویی', 'sickness': 'دلدرد'}

چانک 4:
  متن:     مشورت پزشک مصرف شود.
  متادیتا: {'category': 'راهنمای دارویی', 'sickness': 'دلدرد'}



### 🎯 جمع‌بندی نهایی فصل ۱۳ (پردازش داده‌ها در LangChain):

در این فصل تمامی مهارت‌های لازم برای تبدیل فایل‌های خام به داده‌های آماده برای سیستم‌های RAG را فرا گرفتیم:
1. **آناتومی کلاس `Document`:** تفکیک هوشمندانه `page_content` (متن) و `metadata` (شناسنامه).
2. **انواع لودرها:** استفاده از `TextLoader` برای متون ساده، `PyPDFLoader` برای PDFها با شماره صفحه و `WebBaseLoader` هدفمند با `SoupStrainer`.
3. **معماری Lazy Loading:** بهینه‌سازی مصرف RAM با ژنراتورهای جریانی (`lazy_load`) برای فایل‌های عظیم سازمانی.
4. **استراتژی‌های خرد کردن:** برتری `RecursiveCharacterTextSplitter` بر اسپلیترهای ساده و بومی‌سازی برای زبان فارسی.
5. **حفظ بافتار:** استفاده از `chunk_overlap` (۱۰٪ تا ۲۰٪) برای حل تراژدی ضمیرهای سرگردان و `MarkdownHeaderTextSplitter` برای نگهداری تیترها در متادیتا.

🚀 **گام بعدی:**
اکنون داده‌های ما به صورت قطعات هوشمند و دارای هویت آماده شده‌اند. در **فصل ۱۴ (External Knowledge and RAG)** یاد می‌گیریم که چگونه این قطعات را به یک مدل زبانی (LLM) متصل کنیم تا یک سیستم کامل پرسش‌وپاسخ بر روی اسناد بسازیم.